#File setup

In [ ]:
#Initial setup
#Check GPU availability
import torch
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


CUDA disponibile: True
GPU: Tesla T4
Memoria GPU: 15.83 GB
Mounted at /content/drive


In [ ]:
#Clone Repository
!git clone https://github.com/NazirNayal8/RbA.git
%cd RbA

# check repo structure
!ls -la

# install dependencies
!pip install -q -r requirements.txt
!pip install git+https://github.com/cocodataset/panopticapi.git
!pip install -q tensorboard
!pip install -q matplotlib seaborn
!pip install -q albumentations


Cloning into 'RbA'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 290 (delta 46), reused 40 (delta 36), pack-reused 226 (from 1)
Receiving objects: 100% (290/290), 275.65 KiB | 2.46 MiB/s, done.
Resolving deltas: 100% (119/119), done.
/content/RbA
total 120
drwxr-xr-x 10 root root  4096 Jan  6 16:32 .
drwxr-xr-x  1 root root  4096 Jan  6 16:32 ..
drwxr-xr-x  2 root root  4096 Jan  6 16:32 assets
drwxr-xr-x  7 root root  4096 Jan  6 16:32 ckpts
drwxr-xr-x  5 root root  4096 Jan  6 16:32 configs
drwxr-xr-x  3 root root  4096 Jan  6 16:32 datasets
-rw-r--r--  1 root root 10632 Jan  6 16:32 evaluate_ood.py
drwxr-xr-x  8 root root  4096 Jan  6 16:32 .git
-rw-r--r--  1 root root    57 Jan  6 16:32 .gitattributes
-rw-r--r--  1 root root  3323 Jan  6 16:32 .gitignore
-rw-r--r--  1 root root  2027 Jan  6 16:32 INSTALL.md
-rw-r--r--  1 root root  1068 Jan  6 16:32 LICENSE
drwxr-xr-x  6 root r

# Data Environment & COCO Subset Setup
This section configures the local environment, installs dependencies, and prepares a lightweight version of the COCO dataset for efficient training.

## Directory & Symlink Management
To maintain compatibility with the RbA code without moving Gigabytes of data, we use Symbolic Links (Symlinks):

Cityscapes: Linked to /home/datasets/cityscapes.

COCO: Organized into the standard /home/datasets/COCO/2017/ structure required by the scripts.

## Subset Generation
Instead of processing the full COCO dataset (~19GB), we create a proxy subset:

Filtering: Selects the first 5,000 images from the training set.

Rsync: Efficiently copies only the selected files to a new subset_train2017 folder.

Linking: The training script is redirected to use this subset, significantly reducing data loading overhead.

## Annotation Preparation
Semantic Conversion: Converts COCO panoptic annotations into semantic format.

Binary Masks: Generates the specific binary masks used by the Outlier Exposure (OE) logic to distinguish between known and unknown pixels.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# install panoptic api
!pip install -q git+https://github.com/cocodataset/panopticapi.git

  Preparing metadata (setup.py) ... done


In [ ]:
#install detectron2
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-9ror6shj
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-9ror6shj
  Resolved https://github.com/facebookresearch/detectron2.git to commit fd27788985af0f4ca800bca563acdb700bb890e2
  Preparing metadata (setup.py) ... done


In [ ]:
#install fvcore
!pip install fvcore

In [ ]:
# cd to RbA repo
%cd /content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA

!pip install -r requirements.txt

/content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/RbA


In [ ]:
# dataset directories creation
!mkdir -p /home/datasets
!mkdir -p /content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/coco/subset_train2017

# Symlink Cityscapes
!ln -s /content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/cityscapes \
        /home/datasets/cityscapes

# COCO folders
!mkdir -p /home/datasets/COCO/2017

# Link COCO annotations
!ln -s "/content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/coco/annotations" \
        /home/datasets/COCO/2017/annotations


ln: failed to create symbolic link '/home/datasets/cityscapes/cityscapes': File exists
ln: failed to create symbolic link '/home/datasets/COCO/2017/annotations/annotations': File exists


In [ ]:
# copy subset of 5000 images using rsync
!ls "/content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/coco/train2017" | head -n 5000 > /tmp/filelist.txt
!rsync -ah --progress --files-from=/tmp/filelist.txt \
    "/content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/coco/train2017/" \
    "/content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/coco/subset_train2017/"

Output streaming troncato alle ultime 5000 righe.
000000012551.jpg
         70.95K 100%   36.41MB/s    0:00:00 (xfr#2501, to-chk=2499/5000)
000000012552.jpg
        177.27K 100%  241.11kB/s    0:00:00 (xfr#2502, to-chk=2498/5000)
000000012556.jpg
        127.58K 100%   90.42MB/s    0:00:00 (xfr#2503, to-chk=2497/5000)
000000012567.jpg
        272.72K 100%  389.36kB/s    0:00:00 (xfr#2504, to-chk=2496/5000)
000000012570.jpg
        156.02K 100%  103.16kB/s    0:00:01 (xfr#2505, to-chk=2495/5000)
000000012574.jpg
        189.04K 100%  243.87kB/s    0:00:00 (xfr#2506, to-chk=2494/5000)
000000012597.jpg
        260.97K 100%  174.92kB/s    0:00:01 (xfr#2507, to-chk=2493/5000)
000000012612.jpg
        159.42K 100%  208.69kB/s    0:00:00 (xfr#2508, to-chk=2492/5000)
000000012614.jpg
        278.46K 100%  207.74kB/s    0:00:01 (xfr#2509, to-chk=2491/5000)
000000012620.jpg
        159.53K 100%  267.68kB/s    0:00:00 (xfr#2510, to-chk=2490/5000)
000000012626.jpg
        148.35K 100%  110.85kB/s 

In [ ]:
# update symlink
!rm -rf /home/datasets/COCO/2017/train2017
!ln -s "/content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/coco/subset_train2017" \
        /home/datasets/COCO/2017/train2017

In [ ]:
#only for evaluation
!python /content/drive/MyDrive/AML_Project/fine-tuning_extension/RbA/datasets/prepare_coco_semantic_annos_from_panoptic_annos.py

{1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9, 11: 10, 13: 11, 14: 12, 15: 13, 16: 14, 17: 15, 18: 16, 19: 17, 20: 18, 21: 19, 22: 20, 23: 21, 24: 22, 25: 23, 27: 24, 28: 25, 31: 26, 32: 27, 33: 28, 34: 29, 35: 30, 36: 31, 37: 32, 38: 33, 39: 34, 40: 35, 41: 36, 42: 37, 43: 38, 44: 39, 46: 40, 47: 41, 48: 42, 49: 43, 50: 44, 51: 45, 52: 46, 53: 47, 54: 48, 55: 49, 56: 50, 57: 51, 58: 52, 59: 53, 60: 54, 61: 55, 62: 56, 63: 57, 64: 58, 65: 59, 67: 60, 70: 61, 72: 62, 73: 63, 74: 64, 75: 65, 76: 66, 77: 67, 78: 68, 79: 69, 80: 70, 81: 71, 82: 72, 84: 73, 85: 74, 86: 75, 87: 76, 88: 77, 89: 78, 90: 79, 92: 80, 93: 81, 95: 82, 100: 83, 107: 84, 109: 85, 112: 86, 118: 87, 119: 88, 122: 89, 125: 90, 128: 91, 130: 92, 133: 93, 138: 94, 141: 95, 144: 96, 145: 97, 147: 98, 148: 99, 149: 100, 151: 101, 154: 102, 155: 103, 156: 104, 159: 105, 161: 106, 166: 107, 168: 108, 171: 109, 175: 110, 176: 111, 177: 112, 178: 113, 180: 114, 181: 115, 184: 116, 185: 117, 186: 118, 187: 119, 1

In [ ]:
# Generate binary masks for Outlier Exposure
!python /content/drive/MyDrive/AML_Project/fine-tuning_extension/datasets/meta-ood-master/preparation/prepare_coco_segmentation.py \


loading annotations into memory...
Done (t=30.33s)
creating index...
index created!

Prepare COCO2017 train split for OoD training
Ground truth segmentation mask will be saved in: /home/datasets/COCO/2017/annotations/ood_seg_train2017
Created save directory: /home/datasets/COCO/2017/annotations/ood_seg_train2017
Images Processed: 1978/1978 
Number of created segmentation masks with height and width of at least 480 pixels: 820
FINISHED 00:00:53.61


# Fine-Tuning OE with COCO subset & cityscapes
Implementation of the method with Outlier Exposure provided in the RbA paper


## Path Configuration
- SETUP_PATH: path to setup.by
- TRAINING_SCRIPT: path to train_net.py, script used for fine-tuning/training
- EVAL_SCRIPT: path to evaluate_ood.py, script used for the evaluation of saved models
- TRAIN_CONFIG_FILE_PATH: YAML configuration detailing training specification, hyperparameters
- TRAIN_OUTPUT_PATH: destination folder trained model
- EVAL_OUTPUT_PATH: destination folder evaluation results
- MODELS_PATH: folder with all the saved trained models
- DATASETS_PATH: folder with all the datasets: cityscapes, coco...

Crucial Step: Compiles the CUDA kernels for Multi-Scale Deformable Attention. Ensure you are using a GPU runtime (T4, L4, or A100).

## Training Workflow
To run a fine-tuning experiment:

Run Training: Execute the training cell. Results will be saved in the OUTPUT_DIR.

Resuming: If the session disconnects, use the Resume cell to continue from the last saved checkpoint.

## Evaluation & Results
OOD Evaluation: Runs the evaluate_ood.py script.

Set --selected_models to match your experiment name (e.g., third_conf).

It currently evaluates on the road_anomaly dataset.

Metrics: The results are saved as a .pkl file. The final cell loads and prints the AUPR, AUROC, and FPR95 metrics.

## Troubleshooting & Dataset Management
Dataset Filtering: At the bottom, there is a commented-out sed command. Use this if you need to disable specific OOD datasets (like BDD100K or Fishyscapes) to speed up evaluation or avoid path errors.

Verification: The grep command confirms which datasets are currently active in support.py.



In [27]:
#Mounting drive
from google.colab import drive
import os
drive.mount('/content/drive')

#Founding paths
ROOT_PATH = "/content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension"
RBA_DIR = os.path.join(ROOT_PATH, "RbA")

#available configurations: basic_conf, first_conf, second_conf, third_conf

SETUP_PATH = os.path.join(RBA_DIR, "mask2former/modeling/pixel_decoder/ops/setup.py")
TRAINING_SCRIPT = os.path.join(RBA_DIR, "train_net.py")
EVAL_SCRIPT = os.path.join(RBA_DIR, "evaluate_ood.py")

# Conf and output paths
TRAIN_CONFIG_FILE_PATH = os.path.join(ROOT_PATH, "RbA_experiments/configurations_training/third_conf.yaml")
TRAIN_OUTPUT_PATH = os.path.join(ROOT_PATH, "RbA_experiments/training_results/third_conf")
EVAL_OUTPUT_PATH = os.path.join(ROOT_PATH, "RbA_experiments/evaluations_result")

MODELS_PATH=os.path.join(ROOT_PATH, "RbA_experiments/training_ckpts")
DATASETS_PATH=os.path.join(RBA_DIR,"datasets")

#export
os.environ['SETUP_PATH'] = SETUP_PATH
os.environ['TRAINING_SCRIPT'] = TRAINING_SCRIPT
os.environ['EVAL_SCRIPT'] = EVAL_SCRIPT
os.environ['TRAIN_CONFIG_FILE_PATH'] = TRAIN_CONFIG_FILE_PATH
os.environ['TRAIN_OUTPUT_PATH'] = TRAIN_OUTPUT_PATH
os.environ['EVAL_OUTPUT_PATH'] = EVAL_OUTPUT_PATH
os.environ['MODELS_PATH'] = MODELS_PATH
os.environ['DATASETS_PATH'] = DATASETS_PATH


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Installs the official Panoptic Segmentation API.
# Required for handling COCO-format datasets and calculating Panoptic Quality (PQ) metrics
!pip install git+https://github.com/cocodataset/panopticapi.git

  Cloning https://github.com/cocodataset/panopticapi.git to /tmp/pip-req-build-epnte8vn
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/panopticapi.git /tmp/pip-req-build-epnte8vn
  Resolved https://github.com/cocodataset/panopticapi.git to commit 7bb4655548f98f3fedc07bf37e9040a992b054b0
  Preparing metadata (setup.py) ... done
  Created wheel for panopticapi: filename=panopticapi-0.1-py3-none-any.whl size=8259 sha256=38845b36f23b32d954a6dba881c6caa6ecf2f1b16b63a7185cfd66616c8642ed
  Stored in directory: /tmp/pip-ephem-wheel-cache-62dfn9ez/wheels/bd/60/57/cd1670a1372873f5aa074683f8780c111a67f95d4c17fb08f5
Successfully built panopticapi


In [3]:
%cd $RBA_DIR

!pip install -r requirements.txt

/content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/RbA
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 14.1 MB/s eta 0:00:00
  Created wheel for fairscale: fi

In [2]:
# Detectron2: Base framework for training, data loading, and Mask2Former architecture support.
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-anemyt17
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-anemyt17
  Resolved https://github.com/facebookresearch/detectron2.git to commit fd27788985af0f4ca800bca563acdb700bb890e2
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 3.3 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=7085049 sha256=39c435ce51d6f5c25bac4c92a6730c043b9006525cc07e79063d024bf7696c21
  Stored in d

In [3]:
#setup for training
# Compiles and installs the Multi-Scale Deformable Attention CUDA kernels.
# This is a critical performance component for the Mask2Former backbone used in RbA.
# It allows the model to process high-resolution images with efficient memory usage.
!TORCH_CUDA_ARCH_LIST="7.5" \
 FORCE_CUDA=1 \
 python "$SETUP_PATH" build install

running build
running build_ext
W0113 13:17:03.628000 5130 torch/utils/cpp_extension.py:630] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
W0113 13:17:03.635000 5130 torch/utils/cpp_extension.py:521] The detected CUDA version (12.5) has a minor version mismatch with the version that was used to compile PyTorch (12.6). Most likely this shouldn't be a problem.
W0113 13:17:03.636000 5130 torch/utils/cpp_extension.py:531] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.5
building 'MultiScaleDeformableAttention' extension
creating build/temp.linux-x86_64-cpython-312/content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension/RbA/mask2former/modeling/pixel_decoder/ops/src/cpu
creating build/temp.linux-x86_64-cpython-312/content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension/RbA/mask2former/modeling/pixel_decoder/ops/src/cuda
x86_64-linux-gnu-g++ -fn

In [4]:
# Set the root directory for all training/evaluation datasets.
import os
os.environ["DETECTRON2_DATASETS"] = DATASETS_PATH

In [ ]:
#training

!python "$TRAINING_SCRIPT" \
  --num-gpus 1 \
  --config-file "$TRAIN_CONFIG_FILE_PATH" \
  OUTPUT_DIR "$TRAIN_OUTPUT_PATH" \
  TEST.EVAL_PERIOD 0


Output streaming troncato alle ultime 5000 righe.
/content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/RbA/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/usr/local/lib/python3.12/dist-packages/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/RbA/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/usr/local/lib/python3.12/dist-packages/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.a

In [ ]:
# resume training from checkpoint

!python $TRAINING_SCRIPT \
  --num-gpus 1 \
  --config-file $TRAIN_CONFIG_FILE_PATH \
  --resume \
  OUTPUT_DIR $TRAIN_OUTPUT_PATH  \
  TEST.EVAL_PERIOD 0

Output streaming troncato alle ultime 5000 righe.
  with autocast(enabled=False):
/usr/local/lib/python3.12/dist-packages/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/RbA/mask2former/modeling/matcher.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/usr/local/lib/python3.12/dist-packages/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/RbA/mask2former/modeling/match

In [29]:
#evaluation
#change configuration in selected_models
!python $EVAL_SCRIPT \
  --out_path $EVAL_OUTPUT_PATH \
  --models_folder $MODELS_PATH \
  --datasets_folder $DATASETS_PATH \
  --model_mode selective \
  --selected_models third_conf \
  --dataset_mode selective \
  --selected_datasets road_anomaly

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/drive/.shortcut-targets-by-id/1jdgYbJugyYhSQUzW4Uv-xW_sXNMJs-JD/AML_Project/fine-tuning_extension/RbA/mask2former/modeling/pixel_decoder/msdeformattn.py:323: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
Datasets to be evaluated:
road_anomaly
-----------------------
/content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension/RbA

In [ ]:
#see evaluation result
import pickle

%cd /content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension/RbA_experiments/evaluations_result/basic_conf

with open("results.pkl", "rb") as f:
    data = pickle.load(f)

print(data)

/content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension/RbA_experiments/evaluations_result/basic_conf
{'road_anomaly': {'auroc': np.float64(0.9762458802312577), 'aupr': np.float64(0.8380696428525142), 'fpr95': np.float64(0.08771517003741676)}}


In [ ]:
"""
#comment dataset not required
!sed -i \
    -e '/bdd100k=/s/^/# /' \
    -e '/fishyscapes_laf=/s/^/# /' \
    -e '/fishyscapes_static=/s/^/# /' \
    -e '/lost_and_found=/s/^/# /' \
    -e '/fs_static=/s/^/# /' \
    -e '/cityscapes=/s/^/# /' \
    -e '/cityscapes_c=/s/^/# /' \
    -e 's/^[[:space:]]*fs_static_v2=/        # fs_static_v2=/' \
    -e 's/^[[:space:]]*road_anomaly_21=/        # road_anomaly_21=/' \
    -e 's/^[[:space:]]*road_obstacles=/        # road_obstacles=/' \
    /content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension/RbA/support.py
"""


In [4]:
#Verify
!grep -A 10 "DATASETS = edict" /content/drive/MyDrive/Colab_Notebooks/AML_Project/fine-tuning_extension/RbA/support.py

    DATASETS = edict(
#         cityscapes=Cityscapes(cityscapes_config, transform=transform, split='val', target_type='semantic'),
# #         bdd100k=BDD100KSeg(hparams=bdd100k_config, mode='val', transforms=transform, image_size=(720, 1280)),
        road_anomaly=RoadAnomaly(hparams=road_anomaly_config, transforms=transform),
#         fishyscapes_laf=FishyscapesLAF(hparams=fishyscapes_laf_config, transforms=transform),
#         fs_static=FishyscapesStatic(hparams=fishyscapes_static_config, transforms=transform, version=1),
        # fs_static_v2=FishyscapesStatic(hparams=fishyscapes_static_config, transforms=transform, version=2),
        # road_anomaly_21=RoadAnomaly21(hparams=road_anomaly_21_config, transforms=transform_ra_21),
        # road_obstacles=RoadObstacle21(road_obstacle_21_config, transforms=transform),
#         lost_and_found=LostAndFound(laf_config, transform) 
    )
